Record of a finished experiment (design A, FAIL). The CBR module and case base were removed from the repo because the design did not pass its criteria and is not used by the system. The saved outputs below are the evidence; this notebook no longer runs without src/cbr.py.

# Stage 4: CBR design A, build and evaluation

Follows `docs/cbr_design.md` (issue #9, step 5c). Design B is not built.

Honesty rules used here:

1. The case base, the scaler, the feature weights and all thresholds come from the training split only.
2. Out-of-fold scores come from 5 folds grouped by SIM on the training split (stratified on the label so every fold has anomalies). The fold models use the final model's settings with a fixed 57 trees (best_iteration + 1), so no validation data is used for early stopping.
3. The config is written to disk before validation is looked at. Validation is checked once. The test split is run once at the end and reported whatever the result.
4. A query is never matched to a day of its own SIM.

`CBR_STAGE=train` stops before validation (used for a dry run of the code on training data). The default runs everything.

One departure from the design text: the design asks for a stability check with repeated CV on "train plus validation SIMs". That would reuse validation labels many times, which the step 5c rules forbid, so the stability check below uses training SIMs only.

In [1]:
import os, sys, json, time, warnings
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
import mlflow
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore")
if not os.path.isdir("data") and os.path.isdir(os.path.join("..", "data")):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
from src import cbr

SEED = 42
DATA_DIR, ARTIFACT_DIR = "data", "artifacts"
GATE_LOW, GATE_HIGH = 0.10, 0.90
N_FOLDS, N_TREES = 5, 57
R_GRID = (1.5, 2.0, 3.0)
Q_GRID = np.round(np.arange(0.05, 0.951, 0.05), 2)
N_BOOT = 1000
STAGE = os.environ.get("CBR_STAGE", "full")
FULL = STAGE == "full"

mlflow.set_tracking_uri("sqlite:///mlruns.db")
mlflow.set_experiment("simguard-stage4-cbr")
if mlflow.active_run():
    mlflow.end_run()
run = mlflow.start_run(run_name="cbr_design_a") if FULL else None
print("CWD:", os.getcwd(), "| stage:", STAGE, "| xgboost", xgb.__version__)

C:\Users\ADMIN\Desktop\projects\simguard\ml-service\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/11/02 15:46:54 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\Users\ADMIN\Desktop\projects\simguard\ml-service\venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


CWD: C:\Users\ADMIN\Desktop\projects\simguard\ml-service | stage: full | xgboost 3.4.1


In [2]:
X_train = pd.read_parquet(f"{DATA_DIR}/X_train_v2.parquet")
X_val = pd.read_parquet(f"{DATA_DIR}/X_val_v2.parquet")
X_test = pd.read_parquet(f"{DATA_DIR}/X_test_v2.parquet")
y_train = pd.read_parquet(f"{DATA_DIR}/y_train.parquet")
y_val = pd.read_parquet(f"{DATA_DIR}/y_val.parquet")
y_test = pd.read_parquet(f"{DATA_DIR}/y_test.parquet")
for X, y in ((X_train, y_train), (X_val, y_val), (X_test, y_test)):
    assert (X["sim_id"].values == y["sim_id"].values).all() and (X["date"].values == y["date"].values).all()
feature_cols = json.load(open(f"{ARTIFACT_DIR}/xgboost_feature_columns.json"))
feature_cols = feature_cols["feature_columns"] if isinstance(feature_cols, dict) else feature_cols
assert len(feature_cols) == 42
train_sims, val_sims, test_sims = (set(y["sim_id"]) for y in (y_train, y_val, y_test))
assert not (train_sims & val_sims) and not (train_sims & test_sims) and not (val_sims & test_sims)

model = joblib.load(f"{ARTIFACT_DIR}/xgboost_triage.joblib")
p_val = model.predict_proba(X_val[feature_cols])[:, 1]
p_test = model.predict_proba(X_test[feature_cols])[:, 1]
band_val = (p_val >= GATE_LOW) & (p_val <= GATE_HIGH)
band_test = (p_test >= GATE_LOW) & (p_test <= GATE_HIGH)
yv, yt = y_val["is_anomalous"].values, y_test["is_anomalous"].values
# Band sizes only (labels inside the band are not looked at here).
print(f"band rows  val {band_val.sum():,} | test {band_test.sum():,}")
assert band_val.sum() == 3893 and band_test.sum() == 3887

band rows  val 3,893 | test 3,887


## 1. Out-of-fold scores on the training split

In [3]:
ytr = y_train["is_anomalous"].values
groups = X_train["sim_id"].values
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_of = np.full(len(X_train), -1)
oof_p = np.zeros(len(X_train))
oof_shap = np.zeros((len(X_train), len(feature_cols)))
Xtr = X_train[feature_cols]
for f, (tr, te) in enumerate(sgkf.split(Xtr, ytr, groups)):
    assert not set(groups[tr]) & set(groups[te])
    spw = (ytr[tr] == 0).sum() / (ytr[tr] == 1).sum()
    m = xgb.XGBClassifier(n_estimators=N_TREES, max_depth=6, learning_rate=0.05, subsample=0.8,
                          colsample_bytree=0.8, scale_pos_weight=spw, eval_metric="aucpr",
                          tree_method="hist", random_state=SEED, n_jobs=-1)
    m.fit(Xtr.iloc[tr], ytr[tr], verbose=False)
    fold_of[te] = f
    oof_p[te] = m.predict_proba(Xtr.iloc[te])[:, 1]
    contrib = m.get_booster().predict(xgb.DMatrix(Xtr.iloc[te]), pred_contribs=True, iteration_range=(0, N_TREES))
    oof_shap[te] = contrib[:, :-1]
assert (fold_of >= 0).all()
oof_band = (oof_p >= GATE_LOW) & (oof_p <= GATE_HIGH)
summary = pd.DataFrame({"fold": fold_of, "y": ytr, "band": oof_band}).groupby("fold").agg(
    rows=("y", "size"), anomalies=("y", "sum"), band_rows=("band", "sum"),
    band_anomalies=("y", lambda s: int(s[oof_band[s.index]].sum())))
print(summary.to_string())
print(f"OOF band: {oof_band.sum():,} rows ({oof_band.mean():.2%}), "
      f"{ytr[oof_band].sum()} anomalous, precision {ytr[oof_band].mean():.4f}")

       rows  anomalies  band_rows  band_anomalies
fold                                             
0     12600        238       2754              21
1     12600        247       3295              39
2     12600        258       4039              33
3     12600        275       2717              28
4     12600        274       2775              17
OOF band: 15,580 rows (24.73%), 138 anomalous, precision 0.0089


## 2. Scaler, weights and case base (training only)

In [4]:
scaler_mean, scaler_scale = cbr.fit_scaler(X_train[feature_cols])
weights = cbr.shap_weights(oof_shap)
Z_train = cbr.scale(X_train[feature_cols], scaler_mean, scaler_scale)
w = pd.Series(weights, index=feature_cols).sort_values(ascending=False)
print("Top 10 feature weights (mean |OOF SHAP|, sum 1):")
print(w.head(10).round(4).to_string())

case_base = cbr.build_case_base(X_train[["sim_id", "date"]], Z_train, y_train, oof_p, feature_cols,
                                gate=(GATE_LOW, GATE_HIGH), seed=SEED)
assert set(case_base["sim_id"]) <= train_sims
assert not set(case_base["sim_id"]) & (val_sims | test_sims)
case_path = f"{ARTIFACT_DIR}/{cbr.CASE_BASE_FILE}"
case_base.to_parquet(case_path, index=False)
case_sha = cbr.sha256_file(case_path)
print(case_base.groupby(["label", "source"]).size().to_string())
print(f"case base: {len(case_base):,} cases from {case_base['sim_id'].nunique()} SIMs, "
      f"{os.path.getsize(case_path) / 1024:.0f} KB, sha256 {case_sha[:16]}...")

Top 10 feature weights (mean |OOF SHAP|, sum 1):
peak_mb                       0.3243
cohort_z_iso_anomaly_score    0.1583
hotspot_session_ratio         0.0976
offpeak_ratio                 0.0375
bytes_per_session             0.0327
hotspot_sum_7d                0.0272
hotspot_sessions              0.0271
offpeak_mb                    0.0226
cohort_z_total_mb             0.0184
vol_zscore_14d                0.0182
label      source   
anomalous  anomalous    1292
normal     band         4000
           random       2000
case base: 7,292 cases from 700 SIMs, 1613 KB, sha256 b61e2e8eae2af91a...


## 3. Tuning on the out-of-fold band rows (training only)

In [5]:
tq = np.flatnonzero(oof_band)
t0 = time.perf_counter()
ret_tr = cbr.retrieve(Z_train[tq], X_train["sim_id"].values[tq], case_base, weights, feature_cols)
print(f"retrieval for {len(tq):,} training band rows: {time.perf_counter() - t0:.1f} s")
own = case_base["sim_id"].values[np.concatenate([ret_tr.normal_idx, ret_tr.anomalous_idx], axis=1)]
assert (own != X_train["sim_id"].values[tq][:, None]).all(), "own SIM matched"
a1_tr, a5_tr, n5_tr = cbr.summary_distances(ret_tr)
y_tq, fold_tq, p_tq, sim_tq = ytr[tq], fold_of[tq], oof_p[tq], X_train["sim_id"].values[tq]

def grid_search(a1, a5, n5, y, folds, fold_ids, max_miss=1):
    rows = []
    dc_grid, df_grid = np.quantile(a1, Q_GRID), np.quantile(a5, Q_GRID)
    for r in R_GRID:
        for qc, dc in zip(Q_GRID, dc_grid):
            clear = (a1 > dc) & (n5 <= a5 / r)
            miss = np.array([(clear & (y == 1) & (folds == f)).sum() for f in fold_ids])
            for qf, dfl in zip(Q_GRID, df_grid):
                flag = (a5 < dfl) & (a5 <= n5 / r) & ~clear
                nf = int(flag.sum())
                rows.append(dict(r=r, q_clear=qc, d_clear=float(dc), q_flag=qf, d_flag=float(dfl),
                                 cleared=int(clear.sum()), flagged=nf, resolved=int(clear.sum()) + nf,
                                 misses=int(miss.sum()), max_fold_miss=int(miss.max()),
                                 flag_precision=float(y[flag].mean()) if nf else np.nan))
    g = pd.DataFrame(rows)
    ok = g[g["max_fold_miss"] <= max_miss].copy()
    ok["fp_sort"] = ok["flag_precision"].fillna(-1)
    ok = ok.sort_values(["resolved", "misses", "fp_sort", "d_clear", "d_flag", "r"],
                        ascending=[False, True, False, False, True, False])
    return g, ok

grid, feasible = grid_search(a1_tr, a5_tr, n5_tr, y_tq, fold_tq, range(N_FOLDS))
print(f"{len(grid)} settings, {len(feasible)} meet <= 1 wrongly cleared anomaly in every fold")
cols_show = ["r", "q_clear", "d_clear", "q_flag", "d_flag", "cleared", "flagged", "resolved",
             "misses", "max_fold_miss", "flag_precision"]
print(feasible[cols_show].head(8).round(4).to_string(index=False))
best = feasible.iloc[0]
THRESH = {"d_clear": float(best["d_clear"]), "d_flag": float(best["d_flag"]), "r": float(best["r"])}
print("chosen:", THRESH)

retrieval for 15,580 training band rows: 3.5 s
1083 settings, 380 meet <= 1 wrongly cleared anomaly in every fold
  r  q_clear  d_clear  q_flag  d_flag  cleared  flagged  resolved  misses  max_fold_miss  flag_precision
2.0     0.95   0.9422    0.95  1.3176      270       11       281       1              1          0.9091
3.0     0.05   0.2259    0.90  0.9695      273        6       279       1              1          1.0000
3.0     0.05   0.2259    0.95  1.3176      273        6       279       1              1          1.0000
2.0     0.95   0.9422    0.90  0.9695      270        8       278       1              1          0.8750
3.0     0.10   0.2542    0.90  0.9695      270        6       276       1              1          1.0000
3.0     0.10   0.2542    0.95  1.3176      270        6       276       1              1          1.0000
3.0     0.05   0.2259    0.85  0.8222      273        2       275       1              1          1.0000
3.0     0.15   0.2768    0.90  0.9695      268

### Stability check: 5 times repeated 5-fold CV over training SIMs

In [6]:
stab = []
for rep in range(5):
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=100 + rep)
    folds_rep = np.full(len(tq), -1)
    for f, (_, te) in enumerate(cv.split(np.zeros(len(tq)), y_tq, sim_tq)):
        folds_rep[te] = f
    for hold in range(5):
        tune = folds_rep != hold
        _, ok = grid_search(a1_tr[tune], a5_tr[tune], n5_tr[tune], y_tq[tune], folds_rep[tune],
                            [f for f in range(5) if f != hold])
        b = ok.iloc[0]
        h = ~tune
        dec = cbr.decide(a1_tr[h], a5_tr[h], n5_tr[h], b["d_clear"], b["d_flag"], b["r"])
        flag = dec == "flag"
        stab.append(dict(rep=rep, hold=hold, r=b["r"], q_clear=b["q_clear"], q_flag=b["q_flag"],
                         held_rows=int(h.sum()), held_resolved=float((dec != "escalated").mean()),
                         held_misses=int(((dec == "clear") & (y_tq[h] == 1)).sum()),
                         held_flag_precision=float(y_tq[h][flag].mean()) if flag.any() else np.nan))
stab = pd.DataFrame(stab)
print(stab.groupby(["r", "q_clear", "q_flag"]).size().rename("times chosen").to_string())
print(stab[["held_resolved", "held_misses", "held_flag_precision"]].describe().round(4).to_string())

r    q_clear  q_flag
1.5  0.95     0.95       2
2.0  0.05     0.95       1
     0.90     0.95       3
     0.95     0.95      12
3.0  0.05     0.90       7
       held_resolved  held_misses  held_flag_precision
count        25.0000      25.0000              14.0000
mean          0.0235       0.5200               0.9206
std           0.0179       0.6532               0.2666
min           0.0096       0.0000               0.0000
25%           0.0166       0.0000               1.0000
50%           0.0183       0.0000               1.0000
75%           0.0236       1.0000               1.0000
max           0.1052       2.0000               1.0000


## 4. Freeze the config before looking at validation

In [7]:
config = {
    "design": "A",
    "doc": "docs/cbr_design.md",
    "feature_columns": list(feature_cols),
    "scaler_mean": scaler_mean.tolist(),
    "scaler_scale": scaler_scale.tolist(),
    "weights": weights.tolist(),
    "weights_source": "mean absolute out-of-fold SHAP on the training split, normalised to sum 1",
    "k_per_class": cbr.K_PER_CLASS,
    "per_sim_cap": cbr.PER_SIM_CAP,
    "thresholds": THRESH,
    "threshold_quantiles": {"q_clear": float(best["q_clear"]), "q_flag": float(best["q_flag"])},
    "flags_enabled": True,
    "flag_policy": "flags are switched off if validation flag precision is below 0.5 or there are no flags",
    "gate": {"clear_below": GATE_LOW, "flag_above": GATE_HIGH},
    "oof": {"n_folds": N_FOLDS, "n_trees": N_TREES, "split": "StratifiedGroupKFold by sim_id", "seed": SEED},
    "case_base_file": cbr.CASE_BASE_FILE,
    "case_base_sha256": case_sha,
    "n_cases": {"normal": int((case_base["label"] == "normal").sum()),
                "anomalous": int((case_base["label"] == "anomalous").sum())},
}
cfg_path = f"{ARTIFACT_DIR}/{cbr.CONFIG_FILE}"
with open(cfg_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)
case_base, config = cbr.load(ARTIFACT_DIR)  # reload: sha256 checked
print("config written and reloaded:", config["thresholds"], "| flags_enabled", config["flags_enabled"])

config written and reloaded: {'d_clear': 0.9421928259956901, 'd_flag': 1.3175959316484724, 'r': 2.0} | flags_enabled True


## 5. Evaluation code (band rows only)

In [8]:
def point_metrics(dec, y, p, sim):
    n = len(dec)
    clear, flag, esc = dec == "clear", dec == "flag", dec == "escalated"
    miss = int((clear & (y == 1)).sum())
    anom = int(y.sum())
    # Rival: clear band rows with probability below t, t set so it misses the same number.
    pa = np.sort(p[y == 1])
    t = pa[miss] if miss < len(pa) else np.inf
    rival = int((p < t).sum())
    return dict(band_rows=n, band_anomalies=anom, cleared=int(clear.sum()), flagged=int(flag.sum()),
                share_resolved=float((~esc).mean()), wrongly_cleared=miss,
                recall_in_band=1 - miss / anom if anom else np.nan,
                flag_precision=float(y[flag].mean()) if flag.any() else np.nan,
                remaining=int(esc.sum()), remaining_precision=float(y[esc].mean()) if esc.any() else np.nan,
                rival_threshold=float(min(t, GATE_HIGH)), rival_cleared=rival, rival_share=rival / n,
                share_minus_rival=float((~esc).mean()) - rival / n)

def bootstrap(dec, y, p, sim, n_boot=N_BOOT, seed=SEED):
    sims, inv = np.unique(sim, return_inverse=True)
    clear, flag, esc = dec == "clear", dec == "flag", dec == "escalated"
    order = np.argsort(p, kind="stable")
    p_o, y_o, inv_o = p[order], y[order], inv[order]
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(n_boot):
        c = np.bincount(rng.integers(0, len(sims), len(sims)), minlength=len(sims)).astype(float)
        w = c[inv]
        n = w.sum()
        miss = (w * (clear & (y == 1))).sum()
        nf = (w * flag).sum()
        ne = (w * esc).sum()
        wo = c[inv_o]
        cum_anom = np.cumsum(wo * y_o)
        pos = np.searchsorted(cum_anom, miss, side="right")  # first row where anomalies exceed miss
        rival = wo[:pos].sum()
        if pos < len(p_o):
            rival -= (wo[:pos] * (p_o[:pos] >= p_o[pos])).sum()  # strict "p < t" on ties
        anom = (w * y).sum()
        out.append(dict(share_resolved=1 - ne / n, wrongly_cleared=miss,
                        recall_in_band=1 - miss / anom if anom else np.nan,
                        flag_precision=(w * (flag & (y == 1))).sum() / nf if nf else np.nan,
                        remaining=ne, remaining_precision=(w * (esc & (y == 1))).sum() / ne if ne else np.nan,
                        rival_share=rival / n, share_minus_rival=(1 - ne / n) - rival / n))
    return pd.DataFrame(out)

def report(name, dec, y, p, sim):
    pt = point_metrics(dec, y, p, sim)
    bs = bootstrap(dec, y, p, sim)
    lo, hi = bs.quantile(0.025), bs.quantile(0.975)
    table = pd.DataFrame({"value": pd.Series(pt)})
    table["ci_low"] = pd.Series({k: lo[k] for k in bs.columns})
    table["ci_high"] = pd.Series({k: hi[k] for k in bs.columns})
    print(f"\n{name}")
    print(table.round(4).to_string())
    return pt, table

def verdict(name, pt, table, flags_enabled):
    checks = {
        "at most 1 band anomaly wrongly cleared": pt["wrongly_cleared"] <= 1,
        "share resolved, 95% CI low >= 0.30": table.loc["share_resolved", "ci_low"] >= 0.30,
        "beats rival: CI low of (CBR share - rival share) > 0": table.loc["share_minus_rival", "ci_low"] > 0,
        "flag precision >= 0.5, or flags off": (not flags_enabled) or (pt["flagged"] > 0 and pt["flag_precision"] >= 0.5),
    }
    for k, v in checks.items():
        print(f"  [{'PASS' if v else 'FAIL'}] {k}")
    ok = all(checks.values())
    print(f"{name}: {'PASS' if ok else 'FAIL'}")
    return ok, checks

# Dry run of the same code on the training band (in-sample for the thresholds).
dec_tr = cbr.decide(a1_tr, a5_tr, n5_tr, THRESH["d_clear"], THRESH["d_flag"], THRESH["r"])
_ = report("TRAIN OOF band (thresholds were tuned here, so this is optimistic)", dec_tr, y_tq, p_tq, sim_tq)


TRAIN OOF band (thresholds were tuned here, so this is optimistic)
                          value      ci_low     ci_high
band_rows            15580.0000         NaN         NaN
band_anomalies         138.0000         NaN         NaN
cleared                270.0000         NaN         NaN
flagged                 11.0000         NaN         NaN
share_resolved           0.0180      0.0141      0.0226
wrongly_cleared          1.0000      0.0000      3.0000
recall_in_band           0.9928      0.9733      1.0000
flag_precision           0.9091      0.4000      1.0000
remaining            15299.0000  14816.8000  15781.4500
remaining_precision      0.0083      0.0063      0.0108
rival_threshold          0.1001         NaN         NaN
rival_cleared           77.0000         NaN         NaN
rival_share              0.0049      0.0040      0.1568
share_minus_rival        0.0131     -0.1407      0.0171


## 6. Validation, checked once

In [9]:
results = {}
if FULL:
    q_idx = np.flatnonzero(band_val)
    Zq = cbr.scale(X_val[feature_cols].iloc[q_idx], config["scaler_mean"], config["scaler_scale"])
    ret_v = cbr.retrieve(Zq, X_val["sim_id"].values[q_idx], case_base, np.asarray(config["weights"]), feature_cols)
    a1_v, a5_v, n5_v = cbr.summary_distances(ret_v)
    args_v = (yv[q_idx], p_val[q_idx], X_val["sim_id"].values[q_idx])
    t = config["thresholds"]
    dec_v_full = cbr.decide(a1_v, a5_v, n5_v, t["d_clear"], t["d_flag"], t["r"], True)
    pt_v_full, tab_v_full = report("VALIDATION, full rule (clear and flag)", dec_v_full, *args_v)
    flags_on = pt_v_full["flagged"] > 0 and pt_v_full["flag_precision"] >= 0.5
    print(f"\nflag policy: validation flag precision {pt_v_full['flag_precision']:.4f} "
          f"-> flags_enabled = {flags_on}")
    config["flags_enabled"] = bool(flags_on)
    with open(cfg_path, "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)
    case_base, config = cbr.load(ARTIFACT_DIR)
    if flags_on:
        pt_v, tab_v = pt_v_full, tab_v_full
    else:
        dec_v = cbr.decide(a1_v, a5_v, n5_v, t["d_clear"], t["d_flag"], t["r"], False)
        pt_v, tab_v = report("VALIDATION, final config (clear only)", dec_v, *args_v)
    results["val"] = (pt_v, tab_v)
else:
    print("CBR_STAGE=train: validation skipped")


VALIDATION, full rule (clear and flag)
                         value     ci_low    ci_high
band_rows            3893.0000        NaN        NaN
band_anomalies         42.0000        NaN        NaN
cleared                87.0000        NaN        NaN
flagged                 0.0000        NaN        NaN
share_resolved          0.0223     0.0119     0.0346
wrongly_cleared         0.0000     0.0000     0.0000
recall_in_band          1.0000     1.0000     1.0000
flag_precision             NaN        NaN        NaN
remaining            3806.0000  3619.9250  3987.0500
remaining_precision     0.0110     0.0064     0.0170
rival_threshold         0.1008        NaN        NaN
rival_cleared         128.0000        NaN        NaN
rival_share             0.0329     0.0277     0.5257
share_minus_rival      -0.0105    -0.5099     0.0016

flag policy: validation flag precision nan -> flags_enabled = False

VALIDATION, final config (clear only)
                         value     ci_low    ci_high
band

## 7. Test, run once

In [10]:
if FULL:
    q_idx = np.flatnonzero(band_test)
    Zq = cbr.scale(X_test[feature_cols].iloc[q_idx], config["scaler_mean"], config["scaler_scale"])
    ret_t = cbr.retrieve(Zq, X_test["sim_id"].values[q_idx], case_base, np.asarray(config["weights"]), feature_cols)
    a1_t, a5_t, n5_t = cbr.summary_distances(ret_t)
    t = config["thresholds"]
    dec_t = cbr.decide(a1_t, a5_t, n5_t, t["d_clear"], t["d_flag"], t["r"], config["flags_enabled"])
    results["test"] = report(f"TEST, final config (flags_enabled={config['flags_enabled']})", dec_t,
                             yt[q_idx], p_test[q_idx], X_test["sim_id"].values[q_idx])
else:
    print("CBR_STAGE=train: test skipped")


TEST, final config (flags_enabled=False)
                         value     ci_low    ci_high
band_rows            3887.0000        NaN        NaN
band_anomalies         49.0000        NaN        NaN
cleared                60.0000        NaN        NaN
flagged                 0.0000        NaN        NaN
share_resolved          0.0154     0.0098     0.0227
wrongly_cleared         0.0000     0.0000     0.0000
recall_in_band          1.0000     1.0000     1.0000
flag_precision             NaN        NaN        NaN
remaining            3827.0000  3658.9250  4005.0000
remaining_precision     0.0128     0.0063     0.0208
rival_threshold         0.1029        NaN        NaN
rival_cleared         438.0000        NaN        NaN
rival_share             0.1127     0.1028     0.3479
share_minus_rival      -0.0972    -0.3320    -0.0839


## 8. Worth keeping? (criteria from docs/cbr_design.md)

In [11]:
verdicts = {}
for name, (pt, table) in results.items():
    print()
    verdicts[name] = verdict(name.upper(), pt, table, config["flags_enabled"])[0]
if FULL:
    keep = all(verdicts.values())
    print(f"\nCBR design A worth keeping (both splits PASS): {keep}")


  [PASS] at most 1 band anomaly wrongly cleared
  [FAIL] share resolved, 95% CI low >= 0.30
  [FAIL] beats rival: CI low of (CBR share - rival share) > 0
  [PASS] flag precision >= 0.5, or flags off
VAL: FAIL

  [PASS] at most 1 band anomaly wrongly cleared
  [FAIL] share resolved, 95% CI low >= 0.30
  [FAIL] beats rival: CI low of (CBR share - rival share) > 0
  [PASS] flag precision >= 0.5, or flags off
TEST: FAIL

CBR design A worth keeping (both splits PASS): False


## 9. Retrieval time and evidence example

In [12]:
src_X = X_val if FULL else X_train
idx = np.flatnonzero(band_val)[:1000] if FULL else tq[:1000]
times = []
for _ in range(3):
    t0 = time.perf_counter()
    dec_1k, ev_1k = cbr.run(src_X.iloc[idx], case_base, config)
    times.append(time.perf_counter() - t0)
print(f"scale + retrieve + decide + evidence for 1,000 queries: {min(times):.3f} s (best of 3), "
      f"{[round(x, 3) for x in times]}")
print(json.dumps(ev_1k[0], indent=2))

scale + retrieve + decide + evidence for 1,000 queries: 0.224 s (best of 3), [0.231, 0.23, 0.224]
[
  {
    "sim_id": 907,
    "date": "2025-01-01",
    "label": "normal",
    "misuse_category": "normal",
    "similarity": 0.805822
  },
  {
    "sim_id": 2,
    "date": "2025-01-01",
    "label": "normal",
    "misuse_category": "normal",
    "similarity": 0.731574
  },
  {
    "sim_id": 924,
    "date": "2025-01-02",
    "label": "anomalous",
    "misuse_category": "unauthorized_sharing",
    "similarity": 0.697565
  },
  {
    "sim_id": 148,
    "date": "2025-01-01",
    "label": "normal",
    "misuse_category": "normal",
    "similarity": 0.684227
  },
  {
    "sim_id": 24,
    "date": "2025-01-01",
    "label": "normal",
    "misuse_category": "normal",
    "similarity": 0.673673
  }
]


In [13]:
if FULL:
    mlflow.log_params({"design": "A", "k_per_class": config["k_per_class"], "per_sim_cap": config["per_sim_cap"],
                       "n_normal_cases": config["n_cases"]["normal"], "n_anomalous_cases": config["n_cases"]["anomalous"],
                       "oof_folds": N_FOLDS, "oof_trees": N_TREES, **{k: round(v, 6) for k, v in config["thresholds"].items()},
                       "flags_enabled": config["flags_enabled"], "case_base_sha256": config["case_base_sha256"]})
    for name, (pt, table) in results.items():
        for k, v in pt.items():
            if v is not None and np.isfinite(v):
                mlflow.log_metric(f"{name}_{k}", float(v))
        for k in table.index:
            if np.isfinite(table.loc[k, "ci_low"]):
                mlflow.log_metric(f"{name}_{k}_ci_low", float(table.loc[k, "ci_low"]))
                mlflow.log_metric(f"{name}_{k}_ci_high", float(table.loc[k, "ci_high"]))
        mlflow.log_metric(f"{name}_pass", float(verdicts[name]))
    mlflow.log_metric("retrieval_s_per_1000", min(times))
    mlflow.set_tags({"issue": "9", "step": "5c", "test_runs": "1"})
    mlflow.log_artifact(cfg_path)
    print("MLflow run:", run.info.run_id)
    mlflow.end_run()

MLflow run: 5151094e136b4f4ca14590d52fa7a69e
